# 08 – Discrete-Time Digital PID Control

**Manufacturing context:** Real PLCs and motion controllers run at a fixed scan rate. The controller reads sensors, computes output, and writes to actuators once per sample period — not continuously.

This notebook rewrites PID as a sampled (PLC-style) loop and shows how sample time affects control quality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Editable parameters ----
# Plant (continuous dynamics simulated at fine resolution)
m = 1.0
c = 1.0
k = 4.0
dt_sim = 0.0001   # Fine simulation step [s]

# Controller
Kp = 30.0
Ki = 15.0
Kd = 8.0
Ts = 0.01         # Controller sample time [s]

t_end = 10.0

# Sample-time comparison
Ts_values = [0.005, 0.02, 0.10]

# Multi-setpoint reference trajectory (time, value)
setpoint_schedule = [
    (0.0, 0.0),
    (1.0, 1.0),
    (4.0, 0.5),
    (7.0, 1.5),
    (9.0, 1.0),
]
# -----------------------------

In [ ]:
def make_reference(t, schedule):
    """Build piecewise-constant reference from (time, value) pairs."""
    ref = np.zeros_like(t)
    for t_switch, val in schedule:
        ref[t >= t_switch] = val
    return ref

def simulate_discrete(Ts, ref_signal):
    """Discrete PID controller + continuous plant.
    
    Controller updates every Ts seconds (zero-order hold).
    Plant is integrated at dt_sim resolution.
    """
    t = np.arange(0, t_end, dt_sim)
    n = len(t)
    x = np.zeros(n)
    v = np.zeros(n)
    u = np.zeros(n)

    integral_e = 0.0
    prev_e = 0.0
    u_hold = 0.0
    next_sample = 0.0

    for i in range(1, n):
        # Controller executes at sample instants only
        if t[i] >= next_sample:
            e = ref_signal[i-1] - x[i-1]
            integral_e += e * Ts
            derivative_e = (e - prev_e) / Ts
            u_hold = Kp * e + Ki * integral_e + Kd * derivative_e
            prev_e = e
            next_sample += Ts

        u[i] = u_hold

        # Plant dynamics (continuous, fine step)
        a = (u[i] - c * v[i-1] - k * x[i-1]) / m
        v[i] = v[i-1] + a * dt_sim
        x[i] = x[i-1] + v[i] * dt_sim

    return t, x, u

In [ ]:
# Part 1: Basic discrete control with step input
t = np.arange(0, t_end, dt_sim)
ref_step = np.where(t >= 0.5, 1.0, 0.0)

t, x, u = simulate_discrete(Ts, ref_step)
print(f"Sample time Ts = {Ts}s  ({1/Ts:.0f} Hz)")

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(t, x, label="Output")
axes[0].plot(t, ref_step, "k--", linewidth=0.8, label="Reference")
axes[0].set_ylabel("Position [m]")
axes[0].set_title(f"Discrete PID Control (Ts={Ts}s)")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[1].plot(t, u, color="tab:orange")
axes[1].set_ylabel("Control effort [N]")
axes[1].set_xlabel("Time [s]")
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Part 2: Sample-time comparison
plt.figure(figsize=(8, 4))
for Ts_val in Ts_values:
    _, x_ts, _ = simulate_discrete(Ts_val, ref_step)
    plt.plot(t, x_ts, label=f"Ts={Ts_val}s ({1/Ts_val:.0f} Hz)")
plt.plot(t, ref_step, "k--", linewidth=0.8, label="Reference")
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.title("Effect of Sample Time on Control Quality")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Part 3: Multi-setpoint tracking
ref_multi = make_reference(t, setpoint_schedule)
_, x_multi, u_multi = simulate_discrete(Ts, ref_multi)

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(t, x_multi, label="Output")
axes[0].plot(t, ref_multi, "k--", linewidth=0.8, label="Reference")
axes[0].set_ylabel("Position [m]")
axes[0].set_title("Multi-Setpoint Tracking (PLC-Style Loop)")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[1].plot(t, u_multi, color="tab:orange")
axes[1].set_ylabel("Control effort [N]")
axes[1].set_xlabel("Time [s]")
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### Student Exercise

1. Set `Ts = 0.20` (5 Hz). Does the controller still work? Why or why not?
2. Add a fifth setpoint to `setpoint_schedule` at t=9s.
3. A typical PLC scan rate is 1-10 ms. What `Ts` values in `Ts_values` are realistic for industrial use?

### Suggested Answers

1. With `Ts = 0.20 s` (5 Hz), the controller no longer works well and becomes unstable in this simulation. The sample period is too large relative to the plant dynamics, so the controller reacts too late, the derivative estimate becomes coarse, and the zero-order hold keeps outdated control values applied for too long.

2. A fifth setpoint has been added at `t = 9.0 s` using `(9.0, 1.0)`.

3. From `Ts_values = [0.005, 0.02, 0.10]`, only `0.005 s` (5 ms) is clearly in the typical industrial PLC scan range of 1-10 ms. `0.02 s` (20 ms) is slower than that typical range, and `0.10 s` (100 ms) is not realistic for fast industrial control loops.